# Knowledge Distillation

Train a small **student** to reproduce a large **teacher**'s behaviour, not just its
answers. The student learns from the teacher's full probability distribution — which
encodes *how* the teacher is uncertain — instead of the one-hot label, and that extra
signal is why a distilled small model routinely beats the same architecture trained
from scratch on the same data.

One of four compression axes in this library, and the only one that changes *what the
model learned* rather than how its weights are stored. Compare
[Pruning](pruning.ipynb) (remove weights), [Low-Rank Approximation](low-rank-factorization.ipynb)
(refactor weights) and [Quantization](quantization-gptq-awq.ipynb) (store weights in
fewer bits). They stack: distil first, then quantize the student.

## 1. What & Why

A trained classifier's output is a full distribution, and the ratios among the *wrong*
classes carry real information. A model shown a picture of a lorry might say
`truck 0.85, bus 0.12, car 0.02, dog 0.00001`. The label says only "truck". The
distribution additionally says *this looks somewhat like a bus, nothing like a dog* —
a similarity structure over classes the teacher learned from far more data than the
student will ever see. Hinton called it **dark knowledge**.

**The problem it solves.** You have a model that is accurate and too expensive, and a
smaller architecture that trains to noticeably worse accuracy. Distillation closes much
of that gap without changing the student's architecture or its inference cost — the
teacher is used at *training* time only and then thrown away.

**Reach for it when:**

- You control training and can run the teacher over your data at least once.
- The student has enough capacity to represent the function — distillation improves
  optimisation and generalisation, it does not create capacity that isn't there.
- You have plentiful *unlabelled* in-domain data. This is distillation's quiet
  superpower: the teacher labels it for free, so the transfer set can be far larger
  than your labelled set.

**Don't reach for it when:**

- You cannot run the teacher (API-only model with terms forbidding it — check the
  licence, this is a real constraint on distilling commercial LLMs).
- You need the teacher's accuracy exactly. Distillation narrows the gap; it rarely
  closes it.
- The student is already matching the teacher. Then the bottleneck is data, not
  training signal.

## 2. Mental Model

**A study guide, not an answer key.**

Training on hard labels is studying from an answer key: every question has one right
answer and no indication of what the near-misses were. Training on the teacher's
softened distribution is studying from a marked-up guide by someone who has seen
thousands of these exams: *"the answer is truck; note that bus was tempting here and
dog was never in the running."*

The **temperature** `T` is how much of that marginal commentary you keep. The teacher's
raw distribution is usually near one-hot — a confident model puts 0.9999 on one class,
and the interesting ratios among the rest are crushed to nothing by the exponential.
Dividing logits by `T` before the softmax spreads the mass out and makes those ratios
visible as gradient signal. As `T → ∞` every class converges to uniform and the signal
becomes pure logit-matching; at `T = 1` you are back to the teacher's own confident
output.

That is the whole trick: *raise the temperature to make the teacher's uncertainty
loud enough to learn from.*

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Teacher / student** | The large, accurate source model and the small target model. The teacher is frozen and in eval mode. |
| **Soft targets** | The teacher's output distribution `p_T = softmax(z_teacher / T)`, used in place of (or alongside) the one-hot label. |
| **Temperature `T`** | Logit divisor that flattens the distribution. Typical range 2–10. Applied to **both** teacher and student at training time, and **never** at inference. |
| **Dark knowledge** | The information carried by the relative probabilities of the incorrect classes. |
| **KD loss** | `L = α·T²·KL(p_T ‖ q_T) + (1−α)·CE(q₁, y)` — a weighted blend of matching the teacher and fitting the true label. |
| **The `T²` factor** | Gradients of the softened KL scale as `1/T²`. Multiplying the loss by `T²` keeps the gradient magnitude comparable to the hard-label term so `α` means what you think it means. |
| **Transfer set** | The data the teacher is evaluated on to produce targets. Needs no labels, so it can be much larger than the training set. |
| **Logit matching** | The `T → ∞` limit: regress the student's logits onto the teacher's directly (MSE). Sometimes stronger, and it sidesteps temperature tuning. |
| **Feature / hint distillation** | Also match *intermediate* activations (FitNets), usually via a learned projection since widths differ. |
| **Task-agnostic vs task-specific** | Distil the pretrained backbone once (DistilBERT) versus distil per downstream task. |
| **Sequence-level KD** | For generative models: distil on the teacher's *generated sequences* rather than per-token distributions. The basis of most modern LLM distillation. |
| **Self-distillation** | Teacher and student share an architecture. Still helps — evidence that the effect is partly a regulariser. |

## 4. Setup

The worked examples below are pure NumPy and run on CPU in seconds — the whole
mechanism is small enough to implement honestly rather than describe. The last example
shows the PyTorch/Hugging Face shape and is gated behind an availability check, so the
notebook executes either way.

In [1]:
# %pip install numpy
# Optional, only for the gated Example 4:
# %pip install torch transformers

import numpy as np

print("numpy", np.__version__)

try:
    import torch
    HAVE_TORCH = True
    print("torch", torch.__version__)
except ImportError:
    HAVE_TORCH = False
    print("torch not installed - Example 4 will print its API shape instead of running")

numpy 2.5.1
torch not installed - Example 4 will print its API shape instead of running


## 5. Worked Examples

### Example 1 — what temperature actually does to the signal

The teacher's raw output looks one-hot. Raising `T` is what makes the ratios among the
losing classes large enough to produce gradient.

In [2]:
def softmax(z, T=1.0, axis=-1):
    z = np.asarray(z, dtype=np.float64) / T
    z = z - z.max(axis=axis, keepdims=True)      # numerically stable
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

classes = ["truck", "bus", "car", "cat", "dog"]
teacher_logits = np.array([8.2, 5.1, 3.4, -1.0, -2.7])

print(f"{'T':>5}  " + "  ".join(f"{c:>8}" for c in classes))
for T in (1, 2, 4, 10):
    p = softmax(teacher_logits, T)
    print(f"{T:>5}  " + "  ".join(f"{v:8.4f}" for v in p))

p1, p4 = softmax(teacher_logits, 1), softmax(teacher_logits, 4)
print(f"\nAt T=1 the bottom three classes hold {p1[2:].sum():.5f} of the mass -- "
      f"almost no gradient reaches them.")
print(f"At T=4 they hold {p4[2:].sum():.5f}.")
print(f"\nThe *ratio* bus:car is preserved in log-space, not in probability space:")
print(f"  T=1  bus/car = {p1[1]/p1[2]:6.3f}")
print(f"  T=4  bus/car = {p4[1]/p4[2]:6.3f}   (closer to 1: the distribution is flatter)")

    T     truck       bus       car       cat       dog
    1    0.9493    0.0428    0.0078    0.0001    0.0000
    2    0.7591    0.1611    0.0689    0.0076    0.0033
    4    0.5188    0.2390    0.1562    0.0520    0.0340
   10    0.3239    0.2376    0.2005    0.1291    0.1089

At T=1 the bottom three classes hold 0.00793 of the mass -- almost no gradient reaches them.
At T=4 they hold 0.24226.

The *ratio* bus:car is preserved in log-space, not in probability space:
  T=1  bus/car =  5.474
  T=4  bus/car =  1.530   (closer to 1: the distribution is flatter)


### Example 2 — the KD loss, and why the `T²` factor is not optional

The gradient of the softened KL with respect to the student's logits is
`(1/T)(q_T − p_T)`, so the loss itself carries a `1/T²` scale. Without the `T²`
correction, raising the temperature silently turns the distillation term off and `α`
stops meaning anything.

In [3]:
def kd_loss_and_grad(student_logits, teacher_logits, y, T=4.0, alpha=0.9):
    '''Return (loss, dL/d student_logits) for one example.

    L = alpha * T^2 * KL(p_T || q_T)  +  (1 - alpha) * CE(q_1, y)
    '''
    q_T = softmax(student_logits, T)
    p_T = softmax(teacher_logits, T)
    q_1 = softmax(student_logits, 1.0)

    kl = float(np.sum(p_T * (np.log(p_T + 1e-12) - np.log(q_T + 1e-12))))
    ce = float(-np.log(q_1[y] + 1e-12))
    loss = alpha * T**2 * kl + (1 - alpha) * ce

    onehot = np.zeros_like(q_1)
    onehot[y] = 1.0
    grad = alpha * T * (q_T - p_T) + (1 - alpha) * (q_1 - onehot)
    return loss, grad


student_logits = np.array([2.0, 2.4, 1.1, 0.3, 0.0])
for T in (1.0, 2.0, 4.0, 8.0):
    _, g_scaled = kd_loss_and_grad(student_logits, teacher_logits, y=0, T=T, alpha=1.0)
    g_unscaled = g_scaled / T          # what you get if you forget the T^2 factor
    print(f"T={T:>4}  |grad| with T^2 = {np.abs(g_scaled).sum():6.3f}   "
          f"without = {np.abs(g_unscaled).sum():6.3f}")

print("\nWithout the correction the distillation gradient shrinks as ~1/T: at T=8 it is")
print("an eighth of its T=1 strength, so 'alpha=0.9' would be a lie.")

T= 1.0  |grad| with T^2 =  1.277   without =  1.277
T= 2.0  |grad| with T^2 =  1.942   without =  0.971
T= 4.0  |grad| with T^2 =  2.229   without =  0.557
T= 8.0  |grad| with T^2 =  2.351   without =  0.294

Without the correction the distillation gradient shrinks as ~1/T: at T=8 it is
an eighth of its T=1 strength, so 'alpha=0.9' would be a lie.


### Example 3 — does it actually work? A controlled experiment

The claim to test: with a **small labelled transfer set**, a student trained on the
teacher's soft targets beats the identical student trained on hard labels. Everything
below is multinomial logistic regression trained by full-batch gradient descent, so
teacher and student differ only in the data and signal they see.

In [4]:
rng = np.random.default_rng(0)

# --- synthetic 5-class problem with genuinely overlapping classes -------------
D, K = 20, 5
centres = rng.standard_normal((K, D)) * 1.6

def make(n):
    y = rng.integers(0, K, size=n)
    X = centres[y] + rng.standard_normal((n, D)) * 2.0
    return X, y

X_teacher, y_teacher = make(4000)   # the teacher's large labelled set
X_small,   y_small   = make(60)     # all the labels the student gets
X_test,    y_test    = make(4000)

def train(X, y_or_soft, epochs=600, lr=0.5, soft=False, T=1.0, W=None):
    n, d = X.shape
    W = np.zeros((d, K)) if W is None else W.copy()
    b = np.zeros(K)
    for _ in range(epochs):
        logits = X @ W + b
        if soft:
            # gradient of T^2 * KL(p_T || q_T) w.r.t. logits is T * (q_T - p_T)
            g = T * (softmax(logits, T) - y_or_soft)
        else:
            q = softmax(logits, 1.0)
            onehot = np.zeros_like(q)
            onehot[np.arange(n), y_or_soft] = 1.0
            g = q - onehot
        W -= lr * (X.T @ g) / n
        b -= lr * g.mean(axis=0)
    return W, b

def acc(W, b, X, y):
    return float(((X @ W + b).argmax(1) == y).mean())

teacher = train(X_teacher, y_teacher, epochs=2000, lr=1.0)
print(f"teacher  (4000 labels)              test acc = {acc(*teacher, X_test, y_test):.3f}")

teacher  (4000 labels)              test acc = 0.969


In [5]:
T_KD = 2.0

# The student sees only the 60 examples. Two ways to teach it.
hard = train(X_small, y_small, epochs=2000, lr=1.0)

soft_targets = softmax(X_small @ teacher[0] + teacher[1], T_KD)
distilled = train(X_small, soft_targets, epochs=2000, lr=1.0, soft=True, T=T_KD)

a_teacher = acc(*teacher, X_test, y_test)
a_hard, a_soft = acc(*hard, X_test, y_test), acc(*distilled, X_test, y_test)
print(f"teacher  (4000 hard labels)        test acc = {a_teacher:.3f}")
print(f"student  (60 hard labels)          test acc = {a_hard:.3f}")
print(f"student  (60 soft targets, T={T_KD:g})    test acc = {a_soft:.3f}")
closed = min(1.0, (a_soft - a_hard) / (a_teacher - a_hard))
print(f"\n-> the student closed {closed:.0%} of the gap to its teacher without a single "
      f"extra label\n   (it lands within noise of the teacher, which saw 4000).")
print("\nSame 60 inputs, same architecture, same optimiser. The only difference is that")
print("one of them was told what the teacher thought about the classes it rejected.")

teacher  (4000 hard labels)        test acc = 0.969
student  (60 hard labels)          test acc = 0.929
student  (60 soft targets, T=2)    test acc = 0.970

-> the student closed 100% of the gap to its teacher without a single extra label
   (it lands within noise of the teacher, which saw 4000).

Same 60 inputs, same architecture, same optimiser. The only difference is that
one of them was told what the teacher thought about the classes it rejected.


Now sweep the temperature — and note what happens if you hold the learning rate fixed
while you do it.

In [6]:
print("  T |  lr = 1.0  |  lr = 1.0 / T^2")
for T in (1.0, 2.0, 4.0, 8.0, 20.0):
    st = softmax(X_small @ teacher[0] + teacher[1], T)
    fixed = train(X_small, st, epochs=2000, lr=1.0, soft=True, T=T)
    scaled = train(X_small, st, epochs=2000, lr=1.0 / T**2, soft=True, T=T)
    print(f"{T:>4g} |   {acc(*fixed, X_test, y_test):.3f}    |   "
          f"{acc(*scaled, X_test, y_test):.3f}")
print(f"\nhard-label baseline: {a_hard:.3f}   teacher: {a_teacher:.3f}")

  T |  lr = 1.0  |  lr = 1.0 / T^2
   1 |   0.953    |   0.953
   2 |   0.970    |   0.966


   4 |   0.923    |   0.970


   8 |   0.625    |   0.971


  20 |   0.360    |   0.963

hard-label baseline: 0.929   teacher: 0.969


Read that table carefully, because the obvious conclusion from the left-hand column is
wrong. It looks like distillation collapses above `T = 4` — the usual explanation being
that the soft targets flatten toward uniform and the signal washes out.

That is not what is happening. The right-hand column holds everything else constant and
merely scales the learning rate by `1/T²`, and the collapse disappears entirely: `T=20`
recovers to roughly the same accuracy as `T=2`.

The cause is the `T²` factor from Example 2. It was introduced to keep the distillation
term's gradient *comparable to the hard-label term's* — but it does so by inflating the
gradient, and with a fixed step size an inflated gradient is an inflated **step**. Past
some temperature you are no longer washing out the signal, you are simply diverging.

So the practical rule is that **`T` and the learning rate are not independent
hyperparameters** — changing one without the other conflates "this temperature is wrong"
with "this step size is now too large". Tune them together, or scale `lr` by `1/T²` and
tune `T` alone.

Two things do remain true: `T = 1` is genuinely weaker (the teacher's own output is
nearly one-hot, so there is little dark knowledge to read), and there is no free lunch at
the top end either — at very high `T` with a compensated learning rate you are converging
on plain logit matching, which is a different objective rather than a better one.

### Example 4 — the shape in PyTorch / Hugging Face

Gated so the notebook runs without a GPU or a model download. This is the same loss as
Example 2, expressed the way you would write it in a real training loop.

In [7]:
KD_TRAINING_STEP = '''
import torch.nn.functional as F

def kd_step(student, teacher, batch, T=4.0, alpha=0.9):
    with torch.no_grad():                       # teacher is frozen ...
        t_logits = teacher(**batch).logits      # ... and must be in .eval() mode

    s_logits = student(**batch).logits

    soft = F.kl_div(
        F.log_softmax(s_logits / T, dim=-1),
        F.log_softmax(t_logits / T, dim=-1),
        reduction="batchmean",
        log_target=True,                        # both args already log-probs
    ) * (T ** 2)                                # the correction from Example 2

    hard = F.cross_entropy(s_logits, batch["labels"])
    return alpha * soft + (1 - alpha) * hard
'''

if HAVE_TORCH:
    import torch
    import torch.nn.functional as F

    torch.manual_seed(0)
    t_logits = torch.tensor([[8.2, 5.1, 3.4, -1.0, -2.7]])
    s_logits = torch.tensor([[2.0, 2.4, 1.1, 0.3, 0.0]], requires_grad=True)
    T = 4.0
    soft = F.kl_div(F.log_softmax(s_logits / T, -1),
                    F.log_softmax(t_logits / T, -1),
                    reduction="batchmean", log_target=True) * T**2
    soft.backward()
    print("torch soft-loss :", float(soft))
    print("torch grad      :", s_logits.grad.numpy().round(4))
    ours, g = kd_loss_and_grad(np.array([2.0, 2.4, 1.1, 0.3, 0.0]),
                               teacher_logits, y=0, T=T, alpha=1.0)
    print("our  soft-loss  :", round(ours, 6))
    print("our  grad       :", g.round(4), "  <- same thing")
else:
    print("torch not available; the training step looks like this:")
    print(KD_TRAINING_STEP)

torch not available; the training step looks like this:

import torch.nn.functional as F

def kd_step(student, teacher, batch, T=4.0, alpha=0.9):
    with torch.no_grad():                       # teacher is frozen ...
        t_logits = teacher(**batch).logits      # ... and must be in .eval() mode

    s_logits = student(**batch).logits

    soft = F.kl_div(
        F.log_softmax(s_logits / T, dim=-1),
        F.log_softmax(t_logits / T, dim=-1),
        reduction="batchmean",
        log_target=True,                        # both args already log-probs
    ) * (T ** 2)                                # the correction from Example 2

    hard = F.cross_entropy(s_logits, batch["labels"])
    return alpha * soft + (1 - alpha) * hard



## 6. Gotchas & Pitfalls

- **Forgetting `T²`.** The single most common bug. Your distillation term quietly
  scales as `1/T²` and you conclude distillation "doesn't help". Example 2 shows the
  magnitude.
- **Remembering `T²` and not touching the learning rate.** The other half of the same
  coin, and the subtler failure. The correction inflates the distillation gradient, so
  raising `T` at a fixed step size eventually diverges — and it looks exactly like
  "high temperature destroys the signal". Example 3 separates the two: scale `lr` by
  `1/T²`, or tune `T` and `lr` jointly.
- **Leaving the temperature on at inference.** `T` is a *training-time* device. The
  deployed student uses `T = 1`. Shipping a student that divides its logits by 4 gives
  you a permanently under-confident, badly calibrated model.
- **A teacher in training mode.** Dropout and batch-norm updates in the teacher make
  the targets noisy and non-stationary. Freeze it, `.eval()` it, wrap in `no_grad`.
- **Distilling the teacher's mistakes.** The student faithfully learns the teacher's
  errors and biases, including on out-of-distribution inputs. Blend in the hard-label
  term (`α < 1`) wherever you have real labels; that term is what keeps the student
  anchored to the ground truth.
- **Assuming the transfer set must be labelled.** It must not. Running the teacher over
  a large pile of unlabelled in-domain data is usually a bigger win than tuning `α`.
- **Too small a student.** Below some capacity the student cannot represent the
  function and no amount of signal fixes it. If distillation and from-scratch training
  converge to the same poor accuracy, you are capacity-bound — try
  [pruning a larger model](pruning.ipynb) instead of training a tiny one.
- **Distribution mismatch between transfer and deployment data.** The student only
  learns the teacher's behaviour *where you evaluated the teacher*.
- **Licence terms.** Several commercial model providers explicitly forbid using outputs
  to train competing models. This is a legal constraint, not a technical one, and it
  applies to exactly this technique.
- **Comparing against a weak baseline.** The honest comparison is distilled student vs
  the same student trained from scratch *with the same tuning budget* — not vs an
  untuned one.

## 7. When to Use vs Alternatives

| Approach | What it changes | Typical win | Cost |
|---|---|---|---|
| **Distillation** | Trains a different, smaller model | 2–10× smaller at a few points of accuracy | A full training run + teacher inference over the transfer set |
| [Quantization](quantization-gptq-awq.ipynb) | Weight/activation precision | 2–4× memory, real latency win | Hours; often no retraining at all |
| [Pruning](pruning.ipynb) | Removes weights from the trained model | 1.5–3× if structured | Prune + fine-tune |
| [Low-rank factorization](low-rank-factorization.ipynb) | Refactors weight matrices | Modest, layer-dependent | Cheap, needs fine-tuning |
| [Sparsity induction](sparsity-induction.ipynb) | Trains toward zeros from the start | Higher sparsity than post-hoc | Must own training |

**Rules of thumb.** Quantization first — it is by far the cheapest real win and it
composes with everything else. Reach for distillation when you need an architecture
change (a genuinely smaller or faster model shape), when you own the training pipeline,
and when you have unlabelled in-domain data to exploit. Distillation and quantization
are the standard pairing: distil to a smaller student, then quantize the student.

Distillation is also the only technique here that can *transfer* capability — a student
can be distilled from a teacher with a different architecture entirely, which none of
the weight-surgery methods can do.

## 8. Resources

- [Distilling the Knowledge in a Neural Network](https://arxiv.org/abs/1503.02531) — Hinton, Vinyals & Dean, 2015. The original; short and very readable, and the source of the `T²` argument.
- [DistilBERT, a distilled version of BERT](https://arxiv.org/abs/1910.01108) — the canonical task-agnostic distillation of a pretrained transformer, 40% smaller and 60% faster.
- [FitNets: Hints for Thin Deep Nets](https://arxiv.org/abs/1412.6550) — distilling intermediate representations, not just outputs.
- [Sequence-Level Knowledge Distillation](https://arxiv.org/abs/1606.07947) — the adaptation that makes distillation work for generative sequence models.
- [Hugging Face: knowledge distillation for computer vision](https://huggingface.co/docs/transformers/en/tasks/knowledge_distillation_for_image_classification) — a complete worked `Trainer` implementation.
- [PyTorch: knowledge distillation tutorial](https://docs.pytorch.org/tutorials/beginner/knowledge_distillation_tutorial.html) — end-to-end with both output and feature distillation.